# Catchment Hydrology, MSc course
## Lecture 2 (Europe edition): The runoff ratio of European catchments
*Wouter R. Berghuijs*

This notebook is a European counterpart to `Lecture_2_RunoffRatio.ipynb`, which used the
[CAMELS](https://ral.ucar.edu/solutions/products/camels) dataset for the contiguous United States.
Here we use **EStreams**, a pan-European catalogue of streamflow, hydro-climatic signatures and
landscape descriptors for **17,130 gauged catchments in 41 countries**, spanning roughly 1900–2022
(Nascimento et al., 2024, *Scientific Data*). See the *References* section at the end of this notebook
for the full citation and links to the complete dataset.

The tables bundled in `data/` are the *summary* parts of EStreams: one row per catchment
(`basin_id`) with pre-computed hydro-climatic signatures and static landscape attributes. The full
release also contains daily streamflow and meteorological time series (one file per catchment,
too large to ship inside a Binder repository) — those are available from the archived dataset
linked in the references.

### The runoff ratio

The **runoff ratio** is the fraction of precipitation that leaves a catchment as streamflow:

$$RR = \frac{Q}{P}$$

where $Q$ is mean streamflow and $P$ is mean precipitation (both usually expressed as a long-term
average depth per unit time, e.g. mm/day or mm/year). It is one of the simplest and most
informative catchment "fingerprints": it tells you, on average, what happens to the water that
falls on a catchment, and it is the starting point of the Budyko framework, which relates the
runoff ratio (or its complement, the evaporative index) primarily to aridity.

**Your task:** using the interactive tools below, explore how the runoff ratio of European
catchments varies geographically, and which climatic and landscape characteristics it relates to
most strongly. Work through the numbered questions as you go — they are meant to be discussed, not
answered in one line.

A quick note on data quality: EStreams is compiled from dozens of national data providers with
very different data models, so — like any real, large observational dataset — it contains a small
number of implausible values (e.g. a handful of catchments with a "runoff ratio" of several
thousand, almost certainly caused by unit or area errors upstream in the data chain). The cell
below removes the small number of physically impossible values ($RR<0$ or $RR>5$) so the plots are
readable, but leaves everything else untouched. Keep this in mind: real datasets need this kind of
sanity-checking, and *which* threshold to use is itself a judgment call worth discussing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ipywidgets as widgets
from ipywidgets import interact

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 6)

# ---------------------------------------------------------------------------
# Small helpers shared by every interactive tool below
# ---------------------------------------------------------------------------
def _parse_float(text):
    """Turn a text-box entry into a float, or None if it's blank/invalid ('auto')."""
    try:
        return float(text)
    except (TypeError, ValueError):
        return None


def _apply_limits(ax, xmin='', xmax='', ymin='', ymax=''):
    """Manually override an axis' limits when a value is given; leave auto-scaled otherwise."""
    xmin, xmax, ymin, ymax = (_parse_float(v) for v in (xmin, xmax, ymin, ymax))
    if xmin is not None or xmax is not None:
        cur = ax.get_xlim()
        ax.set_xlim(xmin if xmin is not None else cur[0],
                     xmax if xmax is not None else cur[1])
    if ymin is not None or ymax is not None:
        cur = ax.get_ylim()
        ax.set_ylim(ymin if ymin is not None else cur[0],
                     ymax if ymax is not None else cur[1])


def _fit_and_label(ax, xdata, ydata, fit_type):
    """Fit a curve of the chosen type through (xdata, ydata), draw it, and return its equation
    as a string (or None if no fit could be made)."""
    if fit_type == 'none':
        return None
    x = np.asarray(xdata, dtype=float)
    y = np.asarray(ydata, dtype=float)
    def _poly_str(coeffs, powers):
        # coeffs/powers highest-order first; renders e.g. "2.1x^2 - 0.9x + 1.2" (proper minus signs)
        terms = []
        for coef, power in zip(coeffs, powers):
            label = '' if power == 0 else ('x' if power == 1 else f'x^{power}')
            if not terms:
                terms.append(f'{coef:.3g}{label}')
            else:
                terms.append(f'{"-" if coef < 0 else "+"} {abs(coef):.3g}{label}')
        return ' '.join(terms)

    try:
        if fit_type == 'linear':
            b, a = np.polyfit(x, y, 1)
            xs = np.linspace(x.min(), x.max(), 200)
            ys = b * xs + a
            eq = 'y = ' + _poly_str([b, a], [1, 0])
        elif fit_type == 'quadratic':
            c2, c1, c0 = np.polyfit(x, y, 2)
            xs = np.linspace(x.min(), x.max(), 200)
            ys = c2 * xs**2 + c1 * xs + c0
            eq = 'y = ' + _poly_str([c2, c1, c0], [2, 1, 0])
        elif fit_type == 'cubic':
            c3, c2, c1, c0 = np.polyfit(x, y, 3)
            xs = np.linspace(x.min(), x.max(), 200)
            ys = c3 * xs**3 + c2 * xs**2 + c1 * xs + c0
            eq = 'y = ' + _poly_str([c3, c2, c1, c0], [3, 2, 1, 0])
        elif fit_type == 'exponential':
            mask = y > 0
            b, loga = np.polyfit(x[mask], np.log(y[mask]), 1)
            a = np.exp(loga)
            xs = np.linspace(x[mask].min(), x[mask].max(), 200)
            ys = a * np.exp(b * xs)
            eq = f'y = {a:.3g}*e^({b:.3g}x)'
        elif fit_type == 'logarithmic':
            mask = x > 0
            b, a = np.polyfit(np.log(x[mask]), y[mask], 1)
            xs = np.linspace(x[mask].min(), x[mask].max(), 200)
            ys = b * np.log(xs) + a
            eq = f'y = {b:.3g}*ln(x) + {a:.3g}'
        elif fit_type == 'power':
            mask = (x > 0) & (y > 0)
            b, loga = np.polyfit(np.log(x[mask]), np.log(y[mask]), 1)
            a = np.exp(loga)
            xs = np.linspace(x[mask].min(), x[mask].max(), 200)
            ys = a * xs**b
            eq = f'y = {a:.3g}*x^{b:.3g}'
        else:
            return None
    except Exception:
        return None

    ax.plot(xs, ys, color='black', linewidth=1.5)
    ax.text(0.02, 0.98, eq, transform=ax.transAxes, va='top', ha='left', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='lightgray'))
    return eq


# ---------------------------------------------------------------------------
# Load the EStreams tables (all indexed by the catchment identifier basin_id)
# ---------------------------------------------------------------------------
signatures = pd.read_csv('data/estreams_hydrometeo_signatures.csv')
gauges     = pd.read_csv('data/estreams_gauging_stations.csv')
topography = pd.read_csv('data/estreams_topography_attributes.csv')
soil       = pd.read_csv('data/estreams_soil_attributes.csv')
vegetation = pd.read_csv('data/estreams_vegetation_attributes.csv')
hydrology  = pd.read_csv('data/estreams_hydrology_attributes.csv')
geology    = pd.read_csv('data/estreams_geology_attributes.csv').drop(columns=['lit_dom'])
snowcover  = pd.read_csv('data/estreams_snowcover_attributes.csv')
met_dens   = pd.read_csv('data/estreams_meteorology_density.csv')

# Data-quality step (see markdown cell above): a runoff ratio outside [0, 5] is not physically
# plausible for these catchments and almost certainly reflects an error upstream in the data chain.
signatures.loc[(signatures['q_runoff_ratio'] < 0) | (signatures['q_runoff_ratio'] > 5),
               'q_runoff_ratio'] = np.nan

# Keep only the compact, generally useful columns from the gauging-station metadata
gauges_small = gauges[['basin_id', 'gauge_country', 'lon', 'lat', 'elevation', 'area_estreams']]

# Merge everything into one wide table, one row per catchment
merged = (signatures
          .merge(gauges_small, on='basin_id', how='left')
          .merge(topography,   on='basin_id', how='left')
          .merge(soil,         on='basin_id', how='left')
          .merge(vegetation,   on='basin_id', how='left')
          .merge(hydrology,    on='basin_id', how='left')
          .merge(geology,      on='basin_id', how='left')
          .merge(snowcover,    on='basin_id', how='left')
          .merge(met_dens,     on='basin_id', how='left')
          .set_index('basin_id'))

numeric_vars = merged.select_dtypes(include=[np.number]).columns.tolist()

print(f'{merged.shape[0]:,} catchments, {merged.shape[1]} attributes '
      f'({len(numeric_vars)} numeric), {gauges["gauge_country"].nunique()} countries')
merged.head()


In [ ]:
def run_histogram(variable='q_runoff_ratio', bins=40, xmin='', xmax='', ymin='', ymax=''):
    values = merged[variable].dropna()
    fig, ax = plt.subplots()
    ax.hist(values, bins=bins, color='#3E7CB1', edgecolor='white')
    ax.set_xlabel(variable)
    ax.set_ylabel('number of catchments')
    ax.set_title(f'Distribution of {variable} across {len(values):,} European catchments')
    _apply_limits(ax, xmin, xmax, ymin, ymax)
    plt.show()

    print(values.describe().to_string())
    print(f'skewness: {values.skew():.2f}')

interact(run_histogram,
         variable=widgets.Dropdown(options=numeric_vars, value='q_runoff_ratio',
                                    description='variable:'),
         bins=widgets.IntSlider(min=5, max=150, step=5, value=40, description='bins:'),
         xmin=widgets.Text(value='', description='x min:', placeholder='auto'),
         xmax=widgets.Text(value='', description='x max:', placeholder='auto'),
         ymin=widgets.Text(value='', description='y min:', placeholder='auto'),
         ymax=widgets.Text(value='', description='y max:', placeholder='auto'));


### Questions

**Q1.** What is a typical (e.g. median) runoff ratio for European catchments? How does it compare
to what you know (or can look up) about the CAMELS/US catchments in the original lecture?

**Q2.** Around 1,000 catchments have a runoff ratio greater than 1 (i.e. more water leaves the
catchment than falls on it, on average). Physically, how is that possible? Think about where these
catchments are located (try the map tool below) and what that suggests.

**Q3.** The histogram tool lets you look at the distribution of *any* variable, not just the
runoff ratio. Compare the shape of `aridity` and `frac_snow` — which is more symmetric, and why
might that be?

**Q4.** The data-cleaning step above removed catchments with a runoff ratio above 5. Try changing
that threshold in the code cell (e.g. to 2, or to 20) and re-run. How sensitive are the summary
statistics (mean, median, std) to this choice? Which statistic is most robust to a few extreme
values, and why?

**Q5.** In the scatter tool below you can compute both the Pearson and the Spearman correlation
coefficient. What is the conceptual difference between the two? When would you trust Spearman more
than Pearson for a variable like `q_runoff_ratio`?

In [ ]:
def run_scatter(x='aridity', y='q_runoff_ratio', color_by='none', fit_type='linear',
                 logx=False, logy=False,
                 xmin='', xmax='', ymin='', ymax='', cmin='', cmax=''):
    # de-duplicate: color_by may legitimately be the same variable as x or y
    cols = list(dict.fromkeys([x, y] + ([color_by] if color_by != 'none' else [])))
    data = merged[cols].dropna()
    if logx:
        data = data[data[x] > 0]
    if logy:
        data = data[data[y] > 0]

    fig, ax = plt.subplots()
    if color_by == 'none':
        ax.scatter(data[x], data[y], s=8, alpha=0.35, color='#3E7CB1', edgecolor='none')
    else:
        cvmin, cvmax = _parse_float(cmin), _parse_float(cmax)
        sca = ax.scatter(data[x], data[y], c=data[color_by], cmap='viridis',
                          vmin=cvmin, vmax=cvmax, s=10, alpha=0.6, edgecolor='none')
        plt.colorbar(sca, ax=ax, label=color_by, shrink=0.85)

    eq = _fit_and_label(ax, data[x], data[y], fit_type)

    if logx:
        ax.set_xscale('log')
    if logy:
        ax.set_yscale('log')

    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'n = {len(data):,} catchments')
    _apply_limits(ax, xmin, xmax, ymin, ymax)
    plt.show()

    rho, p_s = stats.spearmanr(data[x], data[y])
    r, p_p = stats.pearsonr(data[x], data[y])
    print(f'Spearman r = {rho:.3f}  (p = {p_s:.1e})')
    print(f'Pearson  r = {r:.3f}  (p = {p_p:.1e})')
    if eq:
        print(f'fit ({fit_type}): {eq}')

interact(run_scatter,
         x=widgets.Dropdown(options=numeric_vars, value='aridity', description='x:'),
         y=widgets.Dropdown(options=numeric_vars, value='q_runoff_ratio', description='y:'),
         color_by=widgets.Dropdown(options=['none'] + numeric_vars, value='none',
                                    description='color by:'),
         fit_type=widgets.Dropdown(
             options=['none', 'linear', 'quadratic', 'cubic', 'exponential', 'logarithmic', 'power'],
             value='linear', description='fit:'),
         logx=widgets.Checkbox(value=False, description='log-scale x'),
         logy=widgets.Checkbox(value=False, description='log-scale y'),
         xmin=widgets.Text(value='', description='x min:', placeholder='auto'),
         xmax=widgets.Text(value='', description='x max:', placeholder='auto'),
         ymin=widgets.Text(value='', description='y min:', placeholder='auto'),
         ymax=widgets.Text(value='', description='y max:', placeholder='auto'),
         cmin=widgets.Text(value='', description='color min:', placeholder='auto'),
         cmax=widgets.Text(value='', description='color max:', placeholder='auto'));


**Q6.** Using the scatter tool, explore which catchment characteristics relate most strongly to
`q_runoff_ratio` — try `aridity`, `p_mean`, `frac_snow`, `ele_mt_mean`, `slp_dg_mean`,
`soil_tawc_mean`, `baseflow_index`, and a few of your own choosing. Which single variable explains
the most variance? Does this match what the Budyko framework predicts (that aridity should be the
dominant control)? Which variables add information *beyond* aridity, and can you give a physical
reason for each?

In [ ]:
try:
    import geopandas as gpd
    # 1:50m resolution — noticeably crisper coastlines/borders than the 110m version, while
    # still small enough (a few MB) to fetch quickly at the start of a Binder session.
    europe = gpd.read_file(
        'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/'
        'ne_50m_admin_0_countries.geojson'
    )
    europe = europe.cx[-25:45, 33:72]  # crop to roughly the EStreams domain
except Exception:
    europe = None  # no internet available right now — the map still works, just without borders

EUROPE_XLIM = (-25, 45)
EUROPE_YLIM = (34, 72)

countries = ['all'] + sorted(merged['gauge_country'].dropna().unique().tolist())

def map_maker(variable='q_runoff_ratio', country='all',
              cmin='', cmax='', xmin='', xmax='', ymin='', ymax=''):
    subset = merged if country == 'all' else merged[merged['gauge_country'] == country]
    data = subset[['lon', 'lat', variable]].dropna()

    fig, ax = plt.subplots(figsize=(13, 12))
    if europe is not None:
        europe.boundary.plot(ax=ax, color='black', linewidth=0.7, zorder=0)

    vmin = _parse_float(cmin)
    vmax = _parse_float(cmax)
    if vmin is None:
        vmin = data[variable].quantile(0.05)
    if vmax is None:
        vmax = data[variable].quantile(0.95)

    sca = ax.scatter(data['lon'], data['lat'], c=data[variable], cmap='viridis',
                      vmin=vmin, vmax=vmax, s=6, zorder=1)
    # a smaller, thinner colorbar so it doesn't dominate the (now larger) map
    plt.colorbar(sca, ax=ax, label=variable, shrink=0.4, fraction=0.035, pad=0.02)
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')
    ax.set_aspect('equal')

    if country == 'all':
        # zoomed to Europe by default, regardless of the extent of the boundary/data
        ax.set_xlim(*EUROPE_XLIM)
        ax.set_ylim(*EUROPE_YLIM)
    else:
        # zoom in further to the selected country, with a small margin
        pad = 1.0
        ax.set_xlim(data['lon'].min() - pad, data['lon'].max() + pad)
        ax.set_ylim(data['lat'].min() - pad, data['lat'].max() + pad)

    # manual overrides (lon/lat limits), applied last so they always win
    _apply_limits(ax, xmin, xmax, ymin, ymax)

    ax.set_title(f'{variable} — {len(data):,} catchments'
                 + ('' if country == 'all' else f' ({country})'))
    plt.show()

interact(map_maker,
         variable=widgets.Dropdown(options=numeric_vars, value='q_runoff_ratio',
                                    description='variable:'),
         country=widgets.Dropdown(options=countries, value='all', description='country:'),
         cmin=widgets.Text(value='', description='color min:', placeholder='auto (p5)'),
         cmax=widgets.Text(value='', description='color max:', placeholder='auto (p95)'),
         xmin=widgets.Text(value='', description='lon min:', placeholder='auto'),
         xmax=widgets.Text(value='', description='lon max:', placeholder='auto'),
         ymin=widgets.Text(value='', description='lat min:', placeholder='auto'),
         ymax=widgets.Text(value='', description='lat max:', placeholder='auto'));


In [ ]:
core_vars = ['q_runoff_ratio', 'q_mean', 'baseflow_index', 'aridity', 'p_mean', 'pet_mean',
             'p_seasonality', 'frac_snow', 'ele_mt_mean', 'slp_dg_mean', 'strm_dens',
             'soil_tawc_mean', 'root_dep_mean', 'lai_mean', 'ndvi_mean', 'sno_cov_mean']
core_vars = [v for v in core_vars if v in merged.columns]

def correlation_overview(method='spearman', cmin='', cmax=''):
    corr = merged[core_vars].corr(method=method)
    vmin = _parse_float(cmin)
    vmax = _parse_float(cmax)
    if vmin is None:
        vmin = -1
    if vmax is None:
        vmax = 1
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                vmin=vmin, vmax=vmax, square=True, ax=ax, cbar_kws=dict(shrink=0.7))
    ax.set_title(f'{method.capitalize()} correlation between core EStreams variables')
    plt.show()

interact(correlation_overview,
         method=widgets.Dropdown(options=['pearson', 'spearman'], value='spearman',
                                  description='method:'),
         cmin=widgets.Text(value='', description='color min:', placeholder='auto (-1)'),
         cmax=widgets.Text(value='', description='color max:', placeholder='auto (1)'));

# Note: this heatmap deliberately uses a curated subset of ~15 variables rather than the full
# ~180-column table — with this many catchments and attributes, a full correlation matrix is both
# unreadable and slow to recompute interactively. Feel free to edit `core_vars` above to add or
# remove variables you're interested in.


### References

Nascimento, T. V. M., Rudlang, J., Höge, M., van der Ent, R., Chappon, M., Seibert, J.,
Hrachowitz, M., & Fenicia, F. (2024). EStreams: An integrated dataset and catalogue of streamflow,
hydro-climatic and landscape variables for Europe. *Scientific Data*, 11, 879.
https://doi.org/10.1038/s41597-024-03706-1

Dataset (Zenodo): https://doi.org/10.5281/zenodo.13154470

Code / tools (Zenodo, mirrored on GitHub at [thiagovmdon/EStreams](https://github.com/thiagovmdon/EStreams)):
https://doi.org/10.5281/zenodo.13255133

The tables used in this notebook are a subset of the full EStreams release (static and temporal
landscape attributes, hydro-climatic signatures, and gauging-station metadata). The full release
also includes catchment boundary shapefiles and per-catchment daily streamflow and meteorological
time series — see the Zenodo dataset above for the complete archive, and please cite the paper and
dataset (and, where relevant, the original national data providers listed in
`streamflow_gauges/estreams_streamflow_catalogue.csv`) in any derived work.

Compare with the original US example: [CatchmentHydro_Lecture2_RunoffRatio](https://github.com/wberghuijs/CatchmentHydro_Lecture2_RunoffRatio)
(Addor et al., 2017; Newman et al., 2015 — CAMELS dataset).